### Импорты

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path

root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.download import download_and_extract_data

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from pathlib import Path
from models.classes.baselines import QuantityBaselineModel, MeanIntervalBaselineModel, EmaIntervalBaselineModel

### Глобальные константы

In [ ]:
dataset_folder_path = Path("./data/Dunnhumby")
files_folder_path = Path("./data/files")

iterations = 300
learning_rate = 1e-3
depth = 2
loss_function = "MAE"
cat_features = ["day_of_week", "session_step"]

### Загрузка данных

In [7]:
dataset_train = pd.read_csv(dataset_folder_path / "dataset_train.csv")
dataset_val = pd.read_csv(dataset_folder_path / "dataset_val.csv")
dataset_test = pd.read_csv(dataset_folder_path / "dataset_test.csv")

In [12]:
X_train, y_train = dataset_train.drop("target", axis=1), dataset_train["target"]
X_val, y_val = dataset_val.drop("target", axis=1), dataset_val["target"]
X_test, y_test = dataset_test.drop("target", axis=1), dataset_test["target"]

In [9]:
dataset_train.columns

Index(['day_of_week', 'session_step', 'current_quantity', 'current_weight',
       'current_volume', 'current_count', 'mean_interval', 'std_interval',
       'ema_interval', 'rolling_mean_interval', 'mean_quantity', 'mean_weight',
       'mean_volume', 'mean_count', 'quantity_lag_1', 'weight_lag_1',
       'volume_lag_1', 'count_lag_1', 'interval_lag_1', 'norm_interval_lag_1',
       'quantity_lag_2', 'weight_lag_2', 'volume_lag_2', 'count_lag_2',
       'interval_lag_2', 'norm_interval_lag_2', 'quantity_lag_3',
       'weight_lag_3', 'volume_lag_3', 'count_lag_3', 'interval_lag_3',
       'norm_interval_lag_3', 'quantity_lag_4', 'weight_lag_4', 'volume_lag_4',
       'count_lag_4', 'interval_lag_4', 'norm_interval_lag_4',
       'quantity_lag_5', 'weight_lag_5', 'volume_lag_5', 'count_lag_5',
       'interval_lag_5', 'norm_interval_lag_5', 'quantity_to_mean_ratio',
       'baseline_pred_by_quantity', 'weight_to_mean_ratio',
       'baseline_pred_by_weight', 'volume_to_mean_ratio',
   

### Обучение моделей

In [13]:
quantity_baseline = QuantityBaselineModel()
meanInterval_baseline = MeanIntervalBaselineModel()
emaInterval_baseline = EmaIntervalBaselineModel()

cb_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=7,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=['day_of_week'],
    random_seed=42,
    verbose=100,
    task_type='CPU'
)

In [14]:
cb_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
    use_best_model=True
)

CatBoostError: Invalid type for cat_feature[non-default value idx=0,feature_idx=0]=4.0 : cat_features must be integer or string, real number values and NaN values should be converted to string.

### Анализ метрик

### Сохранение результатов экспериментов